# Tutorial: P2SA - Practical Example 1 - Access and download Carrington movies

```
Author: ESDC Team at ESAC
Date Last Modified: 15/07/2026
```

This notebook shows how to retrieve and display Carrington movie files directly via TAP using ADQL and the `astroquery` package.

## Requirements

The following python packages are required:


In [ ]:
# python -m pip install astroquery ipython

We use `TapPlus` from `astroquery` to query metadata from the P2SA TAP service. Once we identify a `file_oid`, we build the movie URL and display it in the notebook.


In [2]:
from astroquery.utils.tap.core import TapPlus
from IPython.display import Video

## Access and download Carrington movies with TAP


In this workflow, we query `p2sa.v_carrington_rotation_file` with ADQL to get matching `file_oid` values. We query only for files matching the user-defined `input_date`.
We then build the movie download URL from each `file_oid` and display the movie using `IPython.display.Video`.

Parameters used in the query are:

- `input_date`: mandatory date in format `yyyy-mm-dd`.
- `file_type`: optional (`CR` or `CR_YELLOW`). If omitted, both are queried.


First, define a helper that queries Carrington movie metadata for a single day and returns displayable movie URLs.


In [3]:
p2sa = TapPlus(url='https://p2sa.esac.esa.int/p2sa-sl-tap/tap')

def query_carrington_movie_urls(input_date, file_type=None):
    start = f"{input_date} 00:00:00"

    # Build a one-day window [input_date, input_date + 1 day)
    from datetime import datetime, timedelta
    next_day = datetime.strptime(input_date, '%Y-%m-%d') + timedelta(days=1)
    end = next_day.strftime('%Y-%m-%d 00:00:00')

    if file_type:
        file_type_filter = f"('{file_type}')"
    else:
        file_type_filter = "('CR','CR_YELLOW')"

    query = f"""
    SELECT file_oid
    FROM p2sa.v_carrington_rotation_file
    WHERE (end_date > '{start}')
      AND (start_date < '{end}')
      AND (file_type IN {file_type_filter})
    ORDER BY start_date ASC
    """

    job = p2sa.launch_job(query=query)
    results = job.get_results()

    urls = [
        f"https://p2sa.esac.esa.int/p2sa-sl-tap/video?file_oid={row['file_oid']}&data_retrieval_origin=UI"
        for row in results
    ]
    return urls

movie_urls = query_carrington_movie_urls(input_date='2018-07-02', file_type='CR_YELLOW')

if movie_urls:
    for url in movie_urls:
        display(Video(url, width=400, height=400))
else:
    raise ValueError('No Carrington movies found for the specified date')


As a second step, request the Carrington movie for the previous day to compare timeline context.


In [4]:
movie_urls = query_carrington_movie_urls(input_date='2018-07-01')

if movie_urls:
    for url in movie_urls:
        display(Video(url, width=400, height=400))
else:
    raise ValueError('No Carrington movies found for the specified date')
